In [ ]:
!pip install opencv-contrib-python-headless scipy -q

import cv2, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import time, os, json
from pathlib import Path
from scipy.signal import medfilt
from collections import deque
from google.colab import drive
drive.mount('/content/drive')

HAS_XIMGPROC = False
try:
    import cv2.ximgproc
    _ = cv2.ximgproc.thinning
    HAS_XIMGPROC = True
    print('✅ ximgproc available')
except Exception:
    print('⚠️  ximgproc fallback')

print(f'✅ Ready  |  CUDA:{torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 MB 10.0 MB/s eta 0:00:00
Mounted at /content/drive
✅ ximgproc available
✅ Ready  |  CUDA:True
   GPU: Tesla T4


In [ ]:
TRACKNET_PATH = '/content/drive/MyDrive/In_Out_Pickleball/tracknetv5_5frame_scratch/checkpoints/best_tracknetv5_5frame.pth'
COURTNET_PATH = '/content/drive/MyDrive/In_Out_Pickleball/TennisCourtNet_v2/best.pt'

VIDEO_PATH  = '/content/drive/MyDrive/In_Out_Pickleball/controversial_call/first.mp4'
OUTPUT_PATH = '/content/drive/MyDrive/In_Out_Pickleball/result_in_out/first_codex_ver2_new_algorithm_output_accurate.mp4'
MAX_FRAMES  = None

TN_IMG_W      = 512
TN_IMG_H      = 288
TN_THRESH     = 0.5
TN_BLOB_MIN   = 4
TN_BLOB_MAX   = 1500
TN_MAX_TRAVEL = 200
TN_N_FRAMES       = 5
TN_N_INTERVALS    = TN_N_FRAMES - 1
TN_IN_CH_BACKBONE = TN_N_FRAMES*3 + TN_N_INTERVALS*2
TN_MOTION_ATT_CH  = TN_N_INTERVALS * 2
TN_TSATT_IN_CH    = 1 + TN_MOTION_ATT_CH
TN_HALF           = TN_N_FRAMES // 2

SMOOTH_WINDOW  = 3
SMOOTH_MAX_GAP = 3

COURT_BUFFER_SIZE  = 8
COURT_MOVE_THRESH  = 6
COURT_EMA_ALPHA    = 0.05
COURT_DETECT_EVERY = 10

BOUNCE_COOLDOWN_FRAMES = 8
BOUNCE_VEL_MIN_PX      = 0.8
BOUNCE_APPROACH_MIN_PX = 3.0
V_FIT_WINDOW           = 8
CONTACT_FIT_WINDOW     = 8
BOUNCE_REFINE_MODE     = 'VFIT'
DECISION_MODE          = 'PIXEL_POLYGON'
POLYGON_MARGIN_PX      = 5
COURT_CM_MARGIN        = 5.0
NEAR_LINE_MARGIN_CM    = 3.0
MAX_AERIAL_DIST_PX     = 80
BOUNCE_MIN_SPEED_CM    = 3.0
BOUNCE_DISPLAY_FRAMES  = 45

CONTACT_CORRECTION_JSON = ''
CONTACT_CORRECTION_DEGREE = 2

PARAB_RESIDUE_THRESH   = 100.0
PARAB_HIST_LEN         = 5

GAP_BOUNCE_MAX_FRAMES  = 8
GAP_BOUNCE_VEL_MIN     = 0.8

COURT_X_MIN = 0;   COURT_X_MAX = 610
COURT_Y_MIN = 0;   COURT_Y_MAX = 1372

TRAIL_LEN = 15

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Config  |  Device: {device}')
print(f'   TrackNet : {TRACKNET_PATH}')
print(f'   CourtNet : {COURTNET_PATH}')
print(f'   Algorithm: VSign + PARAB + GAP (3-pass) → V-fit → Pixel Polygon')
print(f'   Bounce   : vel_min={BOUNCE_VEL_MIN_PX}px | cooldown={BOUNCE_COOLDOWN_FRAMES}f | '
      f'gap_max={GAP_BOUNCE_MAX_FRAMES}f | refine={BOUNCE_REFINE_MODE} | '
      f'decision={DECISION_MODE} | vfit_window=±{V_FIT_WINDOW}f | smooth_window={SMOOTH_WINDOW}')

✅ Config  |  Device: cuda
   TrackNet : /content/drive/MyDrive/In_Out_Pickleball/tracknetv5_5frame_scratch/checkpoints/best_tracknetv5_5frame.pth
   CourtNet : /content/drive/MyDrive/In_Out_Pickleball/TennisCourtNet_v2/best.pt
   Algorithm: VSign + PARAB + GAP (3-pass) → V-fit → Pixel Polygon
   Bounce   : vel_min=0.8px | cooldown=8f | gap_max=8f | refine=VFIT | decision=PIXEL_POLYGON | vfit_window=±8f | smooth_window=3


In [ ]:
class MDDLayer5(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(torch.zeros(1))
        self.beta  = nn.Parameter(torch.zeros(1))

    def _attn(self, polarity):
        k = 5.0 / (0.45 * torch.tanh(self.alpha).abs() + 1e-6)
        m = 0.6 * torch.tanh(self.beta)
        return torch.sigmoid(k * (polarity.abs() - m))

    def forward(self, frames):
        imgs = [frames[:, i*3:(i+1)*3] for i in range(TN_N_FRAMES)]
        attn_pairs = []
        for i in range(TN_N_INTERVALS):
            diff_g = (imgs[i+1] - imgs[i]).mean(dim=1, keepdim=True)
            attn_pairs.append(torch.cat([
                self._attn(F.relu( diff_g)),
                self._attn(F.relu(-diff_g)),
            ], dim=1))
        parts = [imgs[0]]
        for i in range(TN_N_INTERVALS):
            parts.append(attn_pairs[i])
            parts.append(imgs[i+1])
        return torch.cat(parts, dim=1), torch.cat(attn_pairs, dim=1)

class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(ConvBnRelu(in_ch, out_ch),
                                    ConvBnRelu(out_ch, out_ch))
    def forward(self, x): return self.block(x)

class V2Backbone5(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1  = DoubleConv(TN_IN_CH_BACKBONE, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2  = DoubleConv(64, 128);   self.pool2 = nn.MaxPool2d(2)
        self.enc3  = DoubleConv(128, 256);  self.pool3 = nn.MaxPool2d(2)
        self.bot   = DoubleConv(256, 512)
        self.up3   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec3  = DoubleConv(512+256, 256)
        self.up2   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec2  = DoubleConv(256+128, 128)
        self.up1   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1  = DoubleConv(128+64, 64)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        return self.dec1(torch.cat([self.up1(d2), e1], dim=1))

class TSATTHead5(nn.Module):
    def __init__(self, patch=8, dim=128, heads=4, layers=2):
        super().__init__()
        self.patch       = patch
        self.patch_embed = nn.Conv2d(TN_TSATT_IN_CH, dim, patch, stride=patch)
        max_tok          = (TN_IMG_H // patch) * (TN_IMG_W // patch)
        self.pos_embed   = nn.Parameter(torch.randn(1, max_tok, dim) * 0.02)
        enc = nn.TransformerEncoderLayer(dim, heads, dim*4, 0.1,
                                          batch_first=True, norm_first=False)
        self.transformer = nn.TransformerEncoder(enc, layers,
                                                  enable_nested_tensor=False)
        self.head = nn.Linear(dim, patch * patch)
    def forward(self, x):
        B, C, H, W = x.shape
        t  = self.patch_embed(x)
        ph, pw = t.shape[2], t.shape[3]
        t  = t.flatten(2).transpose(1, 2)
        t  = t + self.pos_embed[:, :t.shape[1], :]
        t  = self.transformer(t)
        px = self.head(t).transpose(1, 2).reshape(B, self.patch**2, ph, pw)
        return F.pixel_shuffle(px, self.patch)

class RSTR5(nn.Module):
    def __init__(self):
        super().__init__()
        self.draft_conv = nn.Conv2d(64, 1, 1)
        self.tsatt      = TSATTHead5()
        self.dropout    = nn.Dropout2d(0.1)
    def forward(self, feat, motion_attn):
        draft = self.draft_conv(feat)
        inp   = torch.cat([draft, motion_attn], dim=1)
        return torch.sigmoid(draft + self.tsatt(inp))

class TrackNetV5_5frame(nn.Module):
    def __init__(self):
        super().__init__()
        self.mdd      = MDDLayer5()
        self.backbone = V2Backbone5()
        self.rstr     = RSTR5()
    def forward(self, frames):
        Xin, attn = self.mdd(frames)
        return self.rstr(self.backbone(Xin), attn)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1, s=1, bias=True):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, stride=s, padding=p, bias=bias),
            nn.ReLU(), nn.BatchNorm2d(out_ch))
    def forward(self, x): return self.block(x)

class PickleballCourtNet(nn.Module):
    def __init__(self, out_channels=13):
        super().__init__()
        self.conv1=ConvBlock(3,64);    self.conv2=ConvBlock(64,64)
        self.pool1=nn.MaxPool2d(2,2)
        self.conv3=ConvBlock(64,128);  self.conv4=ConvBlock(128,128)
        self.pool2=nn.MaxPool2d(2,2)
        self.conv5=ConvBlock(128,256); self.conv6=ConvBlock(256,256)
        self.conv7=ConvBlock(256,256); self.pool3=nn.MaxPool2d(2,2)
        self.conv8=ConvBlock(256,512); self.conv9=ConvBlock(512,512)
        self.conv10=ConvBlock(512,512)
        self.ups1=nn.Upsample(scale_factor=2)
        self.conv11=ConvBlock(512,256); self.conv12=ConvBlock(256,256)
        self.conv13=ConvBlock(256,256)
        self.ups2=nn.Upsample(scale_factor=2)
        self.conv14=ConvBlock(256,128); self.conv15=ConvBlock(128,128)
        self.ups3=nn.Upsample(scale_factor=2)
        self.conv16=ConvBlock(128,64);  self.conv17=ConvBlock(64,64)
        self.conv18=ConvBlock(64,out_channels)
    def forward(self, x):
        x=self.conv1(x);  x=self.conv2(x);  x=self.pool1(x)
        x=self.conv3(x);  x=self.conv4(x);  x=self.pool2(x)
        x=self.conv5(x);  x=self.conv6(x);  x=self.conv7(x); x=self.pool3(x)
        x=self.conv8(x);  x=self.conv9(x);  x=self.conv10(x)
        x=self.ups1(x);   x=self.conv11(x); x=self.conv12(x); x=self.conv13(x)
        x=self.ups2(x);   x=self.conv14(x); x=self.conv15(x)
        x=self.ups3(x);   x=self.conv16(x); x=self.conv17(x); x=self.conv18(x)
        return x

print("✅ All model architectures defined")
print(f"   TrackNetV5_5frame  : 5-frame input ({TN_N_FRAMES*3}ch)")
print(f"   PickleballCourtNet : 12-keypoint heatmap")

✅ All model architectures defined
   TrackNetV5_5frame  : 5-frame input (15ch)
   PickleballCourtNet : 12-keypoint heatmap


In [ ]:
def preprocess_frame_tn(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (TN_IMG_W, TN_IMG_H))
    return torch.from_numpy(rgb.transpose(2, 0, 1)).float() / 255.0

def extract_ball_center(heatmap_tensor, threshold=TN_THRESH):
    hm      = heatmap_tensor.squeeze().cpu().numpy()
    binary  = (hm >= threshold).astype(np.uint8)
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c    = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < TN_BLOB_MIN or area > TN_BLOB_MAX:
        return None
    mask = np.zeros_like(binary)
    cv2.drawContours(mask, [c], -1, 1, -1)
    weights = hm * mask
    total   = float(weights.sum())
    if total < 1e-6:
        return None
    ys_g, xs_g = np.mgrid[0:hm.shape[0], 0:hm.shape[1]]
    cx = float((xs_g * weights).sum() / total)
    cy = float((ys_g * weights).sum() / total)
    return cx, cy

def model_to_orig(cx, cy, orig_w, orig_h):
    return (int(cx*orig_w/TN_IMG_W), int(cy*orig_h/TN_IMG_H))

def smooth_coords(vals, window=SMOOTH_WINDOW):
    arr = np.array(vals, dtype=np.float64)
    nan_mask = np.isnan(arr)
    if nan_mask.all(): return arr
    arr[nan_mask] = np.interp(np.flatnonzero(nan_mask),
                               np.flatnonzero(~nan_mask), arr[~nan_mask])
    return medfilt(arr, kernel_size=window)

REF_KPS = np.array([
    [  0,1372],[305,1372],[610,1372],
    [610, 915],[305, 915],[  0, 915],
    [  0, 457],[305, 457],[610, 457],
    [610,   0],[305,   0],[  0,   0],
], dtype=np.float32)

CONFIGS = [
    [0,2,9,11],[0,2,6,8],[0,2,3,5],[6,8,9,11],
    [0,5,6,11],[2,3,8,9],[1,7,10,4],[0,1,9,10],
    [1,2,10,9],[0,2,1,10],[3,5,6,8],[0,11,2,9],
]

SKELETON = [
    (0,1),(1,2),(3,4),(4,5),(6,7),(7,8),(9,10),(10,11),
    (0,5),(2,3),(5,6),(3,8),(6,11),(8,9),(1,4),(4,7),(7,10),
]

def skeletonize(binary):
    if HAS_XIMGPROC:
        try:
            b = (binary>0).astype(np.uint8)*255
            return cv2.ximgproc.thinning(b, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)
        except: pass
    binary_u8 = (binary>0).astype(np.uint8)*255
    skeleton = np.zeros_like(binary_u8)
    element  = cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3))
    temp     = binary_u8.copy()
    while True:
        eroded   = cv2.erode(temp, element)
        opened   = cv2.dilate(eroded, element)
        skeleton = cv2.bitwise_or(skeleton, cv2.subtract(temp, opened))
        temp     = eroded.copy()
        if cv2.countNonZero(temp) == 0: break
    return skeleton

def refine_kps_cv(frame_bgr, kps, crop_radius=40):
    H_f, W_f = frame_bgr.shape[:2]
    refined_kps  = kps.copy().astype(float)
    refined_mask = np.zeros(len(kps), dtype=bool)
    for k in range(len(kps)):
        px, py = kps[k]
        if px < 0 or py < 0: continue
        cx = int(round(px)); cy = int(round(py))
        cx1=max(0,cx-crop_radius); cy1=max(0,cy-crop_radius)
        cx2=min(W_f-1,cx+crop_radius); cy2=min(H_f-1,cy+crop_radius)
        if (cx2-cx1)<10 or (cy2-cy1)<10: continue
        crop = frame_bgr[cy1:cy2, cx1:cx2]
        hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        white_mask = cv2.inRange(hsv, np.array([0,0,160]),  np.array([180,60,255]))
        green_mask = cv2.inRange(hsv, np.array([30,30,30]), np.array([90,255,255]))
        white_mask[green_mask > 0] = 0
        skeleton = skeletonize(white_mask)
        lines = cv2.HoughLinesP(skeleton, 1, np.pi/180, threshold=15,
                                 minLineLength=10, maxLineGap=10)
        if lines is None or len(lines) < 2: continue
        h_lines, v_lines = [], []
        for ln in lines:
            lx1,ly1,lx2,ly2 = ln[0]
            angle = abs(np.degrees(np.arctan2(ly2-ly1, lx2-lx1)))
            if angle<30 or angle>150: h_lines.append(ln[0])
            elif 60<angle<120:        v_lines.append(ln[0])
        if not h_lines or not v_lines: continue
        ccx, ccy = crop.shape[1]//2, crop.shape[0]//2
        best_pt, best_d = None, float('inf')
        for hl in h_lines:
            for vl in v_lines:
                hl_x1,hl_y1,hl_x2,hl_y2 = hl
                vl_x1,vl_y1,vl_x2,vl_y2 = vl
                denom = (hl_x1-hl_x2)*(vl_y1-vl_y2)-(hl_y1-hl_y2)*(vl_x1-vl_x2)
                if abs(denom) < 1e-6: continue
                t  = ((hl_x1-vl_x1)*(vl_y1-vl_y2)-(hl_y1-vl_y1)*(vl_x1-vl_x2))/denom
                ix = hl_x1+t*(hl_x2-hl_x1); iy = hl_y1+t*(hl_y2-hl_y1)
                if not (0<=ix<crop.shape[1] and 0<=iy<crop.shape[0]): continue
                xi, yi = int(round(ix)), int(round(iy))
                roi = white_mask[max(0,yi-3):yi+4, max(0,xi-3):xi+4]
                if not np.any(roi > 0): continue
                d = np.hypot(ix-ccx, iy-ccy)
                if d < best_d: best_d, best_pt = d, (ix, iy)
        if best_pt is not None:
            rx = cx1+best_pt[0]; ry = cy1+best_pt[1]
            if np.hypot(rx-cx, ry-cy) < crop_radius*0.9:
                refined_kps[k] = [rx, ry]; refined_mask[k] = True
    return refined_kps, refined_mask

def get_best_homography(kps):
    visible = {k: kps[k] for k in range(12)
               if float(kps[k][0])>=0 and float(kps[k][1])>=0}
    if len(visible) < 4: return None
    best_H, best_err = None, float('inf')
    for conf in CONFIGS:
        if not all(c in visible for c in conf): continue
        src = np.float32([REF_KPS[c] for c in conf])
        dst = np.float32([kps[c]     for c in conf])
        H, _ = cv2.findHomography(src, dst, method=0)
        if H is None: continue
        proj = cv2.perspectiveTransform(REF_KPS.reshape(-1,1,2), H).reshape(-1,2)
        errs = [np.hypot(float(visible[k][0])-proj[k][0],
                         float(visible[k][1])-proj[k][1])
                for k in visible if k not in conf]
        if not errs: continue
        mean_err = np.mean(errs)
        if mean_err < best_err: best_err, best_H = mean_err, H
    return best_H

def apply_homography_correction(kps, H_mat, threshold=15):
    if H_mat is None: return kps.copy()
    proj = cv2.perspectiveTransform(REF_KPS.reshape(-1,1,2), H_mat).reshape(-1,2)
    final = kps.copy().astype(float)
    for k in range(12):
        if float(kps[k][0])<0 or float(kps[k][1])<0:
            final[k] = proj[k]
        else:
            dist = np.hypot(float(kps[k][0])-proj[k][0], float(kps[k][1])-proj[k][1])
            if dist > threshold: final[k] = proj[k]
    return final

def predict_court_kps(court_model, frame_bgr, device):
    H_orig, W_orig = frame_bgr.shape[:2]
    MODEL_H, MODEL_W = 360, 640
    img = cv2.resize(frame_bgr, (MODEL_W, MODEL_H))
    inp = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
    inp = torch.FloatTensor(inp.transpose(2,0,1)).unsqueeze(0).to(device)
    court_model.eval()
    with torch.no_grad(): hm = court_model(inp)
    hm_np = hm[0, :12].cpu().numpy()
    kps   = np.zeros((12,2), dtype=np.float32)
    for k in range(12):
        idx    = int(hm_np[k].argmax())
        kps[k] = [int(idx%MODEL_W)*W_orig/MODEL_W,
                  int(idx//MODEL_W)*H_orig/MODEL_H]
    kps, _ = refine_kps_cv(frame_bgr, kps, crop_radius=40)
    H_mat  = get_best_homography(kps)
    kps    = apply_homography_correction(kps, H_mat)
    return kps, H_mat

class KPStabilizer:
    def __init__(self, buffer_size=8, move_thresh=6, ema_alpha=0.05):
        self.buffer      = deque(maxlen=buffer_size)
        self.move_thresh = move_thresh
        self.ema_alpha   = ema_alpha
        self.locked      = None
    def update(self, kps):
        self.buffer.append(kps.copy())
        median_kps = (np.median(np.stack(list(self.buffer), axis=0), axis=0)
                      if len(self.buffer) >= 3 else kps.copy())
        if self.locked is None:
            self.locked = median_kps.copy().astype(float)
        else:
            for k in range(12):
                dist = np.hypot(median_kps[k][0]-self.locked[k][0],
                                median_kps[k][1]-self.locked[k][1])
                if dist > self.move_thresh:
                    self.locked[k][0] = (self.ema_alpha*median_kps[k][0] +
                                         (1-self.ema_alpha)*self.locked[k][0])
                    self.locked[k][1] = (self.ema_alpha*median_kps[k][1] +
                                         (1-self.ema_alpha)*self.locked[k][1])
        return self.locked.copy()

def build_pixel_to_court_homography(kps_in_image):
    src_pts, dst_pts = [], []
    for k in range(12):
        x, y = kps_in_image[k]
        if x < 0 or y < 0: continue
        src_pts.append([float(x), float(y)])
        dst_pts.append([float(REF_KPS[k][0]), float(REF_KPS[k][1])])
    if len(src_pts) < 4: return None
    src = np.float32(src_pts); dst = np.float32(dst_pts)
    H, _ = cv2.findHomography(src, dst, method=cv2.RANSAC, ransacReprojThreshold=10.0)
    if H is None:
        H, _ = cv2.findHomography(src, dst, method=0)
    return H

def pixel_to_court_cm(x_px, y_px, H):
    if H is None: return None, None
    try:
        pt = np.array([[[float(x_px), float(y_px)]]], dtype=np.float32)
        r  = cv2.perspectiveTransform(pt, H)
        return float(r[0,0,0]), float(r[0,0,1])
    except: return None, None

def court_coords_are_sane(x_cm, y_cm, margin=100):
    if x_cm is None or y_cm is None: return False
    return ((-margin <= x_cm <= COURT_X_MAX+margin) and
            (-margin <= y_cm <= COURT_Y_MAX+margin))

def is_ball_in(x_cm, y_cm):
    if x_cm is None or y_cm is None: return None
    return (COURT_X_MIN <= x_cm <= COURT_X_MAX and
            COURT_Y_MIN <= y_cm <= COURT_Y_MAX)

def court_cm_decision(x_cm, y_cm, margin=COURT_CM_MARGIN, near_margin=NEAR_LINE_MARGIN_CM):
    if x_cm is None or y_cm is None:
        return 'UNKNOWN'
    inside_margin = (COURT_X_MIN - margin <= x_cm <= COURT_X_MAX + margin and
                     COURT_Y_MIN - margin <= y_cm <= COURT_Y_MAX + margin)
    if not inside_margin:
        return 'OUT'
    inside_strict = (COURT_X_MIN + near_margin < x_cm < COURT_X_MAX - near_margin and
                     COURT_Y_MIN + near_margin < y_cm < COURT_Y_MAX - near_margin)
    if inside_strict:
        return 'IN'
    true_inside = (COURT_X_MIN <= x_cm <= COURT_X_MAX and
                   COURT_Y_MIN <= y_cm <= COURT_Y_MAX)
    near_outer_line = (COURT_X_MIN - near_margin <= x_cm <= COURT_X_MAX + near_margin and
                       COURT_Y_MIN - near_margin <= y_cm <= COURT_Y_MAX + near_margin)
    return 'UNCERTAIN' if (true_inside or near_outer_line) else 'OUT'

def _poly_features_xy(x, y, degree=2):
    feats = [1.0, float(x), float(y)]
    if degree >= 2:
        feats.extend([float(x)*float(x), float(x)*float(y), float(y)*float(y)])
    return np.array(feats, dtype=np.float64)

def _fit_contact_corrector(samples, degree=2):
    rows, dxs, dys = [], [], []
    for s in samples:
        raw_x = s.get('raw_x_cm', s.get('x_raw_cm', s.get('x_cm')))
        raw_y = s.get('raw_y_cm', s.get('y_raw_cm', s.get('y_cm')))
        ct_x  = s.get('contact_x_cm', s.get('x_contact_cm'))
        ct_y  = s.get('contact_y_cm', s.get('y_contact_cm'))
        if raw_x is None or raw_y is None or ct_x is None or ct_y is None:
            continue
        rows.append(_poly_features_xy(raw_x, raw_y, degree))
        dxs.append(float(ct_x) - float(raw_x))
        dys.append(float(ct_y) - float(raw_y))
    if len(rows) < 3:
        return None
    X = np.vstack(rows)
    return {
        'degree': degree,
        'coeff_x': np.linalg.lstsq(X, np.array(dxs), rcond=None)[0].tolist(),
        'coeff_y': np.linalg.lstsq(X, np.array(dys), rcond=None)[0].tolist(),
    }

def load_contact_corrector(path=CONTACT_CORRECTION_JSON):
    if not path:
        return None
    try:
        with open(path, 'r') as f:
            data = json.load(f)
        if isinstance(data, dict) and 'coeff_x' in data and 'coeff_y' in data:
            return data
        if isinstance(data, list):
            return _fit_contact_corrector(data, CONTACT_CORRECTION_DEGREE)
    except Exception as e:
        print(f'⚠️  Contact correction disabled: {e}')
    return None

def apply_contact_correction(x_cm, y_cm, corrector):
    if x_cm is None or y_cm is None:
        return x_cm, y_cm
    if corrector is None:
        return x_cm, y_cm
    degree = int(corrector.get('degree', CONTACT_CORRECTION_DEGREE))
    feats = _poly_features_xy(x_cm, y_cm, degree)
    cx = np.array(corrector.get('coeff_x', []), dtype=np.float64)
    cy = np.array(corrector.get('coeff_y', []), dtype=np.float64)
    if len(cx) != len(feats) or len(cy) != len(feats):
        return x_cm, y_cm
    return float(x_cm + feats.dot(cx)), float(y_cm + feats.dot(cy))

COURT_CORNER_IDX = [11, 9, 2, 0]

def build_court_polygon(kps):
    corners = []
    for k in COURT_CORNER_IDX:
        cx, cy = float(kps[k][0]), float(kps[k][1])
        if cx >= 0 and cy >= 0:
            corners.append([cx, cy])
    if len(corners) < 4:
        return None
    return np.array(corners, dtype=np.float32)

def pixel_polygon_inout(x_px, y_px, polygon):
    if polygon is None:
        return 'UNKNOWN', None
    dist = cv2.pointPolygonTest(
        polygon, (float(x_px), float(y_px)), measureDist=True)
    dist = float(dist)
    if dist < -MAX_AERIAL_DIST_PX:
        return 'UNKNOWN', dist
    elif dist > POLYGON_MARGIN_PX:
        return 'IN', dist
    elif dist < -POLYGON_MARGIN_PX:
        return 'OUT', dist
    else:
        return 'UNCERTAIN', dist

class BounceDetector:

    def __init__(self, raw_positions=None, contact_corrector=None,
                 refine_mode=BOUNCE_REFINE_MODE, decision_mode=DECISION_MODE):
        self.raw_positions = raw_positions
        self.contact_corrector = contact_corrector
        self.refine_mode = str(refine_mode).upper()
        self.decision_mode = str(decision_mode).upper()

    def detect_bounces(self, smoothed, court_kps_all, court_Hpx2ct_all=None):
        N = len(smoothed)
        smooth_positions = [
            (i, float(smoothed[i][0]), float(smoothed[i][1]))
            for i in range(N) if smoothed[i] is not None
        ]

        vel_candidates   = self._vel_pass(smooth_positions)
        parab_candidates = self._parab_residue_pass(smooth_positions)
        gap_candidates   = self._gap_bounce_pass(smoothed)
        candidates       = self._merge_candidates(vel_candidates, gap_candidates, parab_candidates)

        events = []
        for cand in candidates:
            bf         = cand['frame']
            src_method = cand['method']

            if self.refine_mode == 'CONTACTFIT':
                fit = self._contact_fit_near_frame(bf, smoothed)
                tag = 'CONTACTFIT'
            else:
                window = [
                    (fi, fx, fy) for fi, fx, fy in smooth_positions
                    if bf - V_FIT_WINDOW <= fi <= bf + V_FIT_WINDOW
                ]
                fit = self._vfit_on_window(window)
                tag = 'VFIT'

            if fit is not None:
                x_px, y_px, frame = fit['x_px'], fit['y_px'], fit['frame']
                method = src_method + '+' + tag
            else:
                x_px, y_px, frame = cand['x_px'], cand['y_px'], cand['frame']
                method = src_method + '+POS'

            decision_px, dist_px = 'UNKNOWN', None
            kps = court_kps_all[min(frame, N - 1)]
            if kps is not None:
                polygon = build_court_polygon(kps)
                if polygon is not None:
                    decision_px, dist_px = pixel_polygon_inout(x_px, y_px, polygon)

            raw_x_cm = raw_y_cm = x_cm = y_cm = None
            if court_Hpx2ct_all is not None:
                H = court_Hpx2ct_all[min(frame, N - 1)]
                if H is not None:
                    raw_x_cm, raw_y_cm = pixel_to_court_cm(x_px, y_px, H)
                    x_cm, y_cm = apply_contact_correction(raw_x_cm, raw_y_cm, self.contact_corrector)

            if self.decision_mode == 'COURT_CM':
                decision = court_cm_decision(x_cm, y_cm)
                if decision == 'UNKNOWN':
                    decision = decision_px
            else:
                decision = decision_px

            events.append({
                'frame'       : frame,
                'x_px'        : x_px,
                'y_px'        : y_px,
                'dist_px'     : dist_px,
                'decision_px' : decision_px,
                'decision'    : decision,
                'raw_x_cm'    : raw_x_cm,
                'raw_y_cm'    : raw_y_cm,
                'x_cm'        : x_cm,
                'y_cm'        : y_cm,
                'method'      : method,
            })
        return events

    def _vel_pass(self, positions):
        n = len(positions)
        if n < 5: return []
        frames_arr = np.array([p[0] for p in positions], dtype=float)
        xs_arr     = np.array([p[1] for p in positions], dtype=float)
        ys_arr     = np.array([p[2] for p in positions], dtype=float)
        vy = np.diff(ys_arr)
        vel_window        = 2
        min_speed         = float(BOUNCE_VEL_MIN_PX)
        approach_min      = float(BOUNCE_APPROACH_MIN_PX)
        last_bounce_frame = -9999
        candidates        = []
        for i in range(vel_window, len(vy) - vel_window):
            avg_before = float(np.mean(vy[i - vel_window : i]))
            avg_after  = float(np.mean(vy[i + 1 : i + vel_window + 1]))
            delta_vy   = avg_before - avg_after
            primary_near  = avg_before >= approach_min and avg_after <= -min_speed
            fallback_near = avg_before > 0 and avg_after < 0 and delta_vy >= approach_min * 2
            decel_near = (avg_before >= min_speed * 8 and avg_after >= 0
                          and avg_after < min_speed and delta_vy >= min_speed * 8)
            if primary_near or fallback_near or decel_near:
                bounce_idx   = i + 1
                bounce_frame = int(round(float(frames_arr[bounce_idx])))
                if bounce_frame - last_bounce_frame < BOUNCE_COOLDOWN_FRAMES:
                    continue
                candidates.append({
                    'frame' : bounce_frame,
                    'x_px'  : float(xs_arr[bounce_idx]),
                    'y_px'  : float(ys_arr[bounce_idx]),
                    'method': 'VEL',
                })
                last_bounce_frame = bounce_frame
        return candidates

    def _parab_residue_pass(self, positions):
        n = len(positions)
        if n < PARAB_HIST_LEN + 1:
            return []
        candidates        = []
        last_bounce_frame = -9999
        for i in range(PARAB_HIST_LEN, n):
            window_pts = positions[i - PARAB_HIST_LEN : i]
            frames = np.array([p[0] for p in window_pts], dtype=float)
            ys     = np.array([p[2] for p in window_pts], dtype=float)
            vy_hist = np.diff(ys)
            if not np.any(vy_hist > 1.0):
                continue
            try:
                coeffs = np.polyfit(frames, ys, 2)
                y_pred = np.polyval(coeffs, frames)
                mse    = float(np.mean((ys - y_pred) ** 2))
            except Exception:
                continue
            if mse < PARAB_RESIDUE_THRESH:
                continue
            bounce_frame = int(round(float(positions[i - 1][0])))
            if bounce_frame - last_bounce_frame < BOUNCE_COOLDOWN_FRAMES:
                continue
            candidates.append({
                'frame' : bounce_frame,
                'x_px'  : float(positions[i - 1][1]),
                'y_px'  : float(positions[i - 1][2]),
                'method': 'PARAB',
            })
            last_bounce_frame = bounce_frame
        return candidates

    def _gap_bounce_pass(self, smoothed):
        N = len(smoothed)
        candidates        = []
        last_bounce_frame = -9999
        i = 0
        while i < N:
            if smoothed[i] is not None:
                i += 1
                continue
            gap_start = i
            while i < N and smoothed[i] is None:
                i += 1
            gap_end = i - 1
            gap_len = gap_end - gap_start + 1
            if gap_len > GAP_BOUNCE_MAX_FRAMES:
                continue
            pre_pts = []
            for j in range(gap_start - 1, max(-1, gap_start - 6), -1):
                if j >= 0 and smoothed[j] is not None:
                    pre_pts.append((j, float(smoothed[j][0]), float(smoothed[j][1])))
                if len(pre_pts) >= 3:
                    break
            pre_pts.reverse()
            post_pts = []
            for j in range(gap_end + 1, min(N, gap_end + 6)):
                if smoothed[j] is not None:
                    post_pts.append((j, float(smoothed[j][0]), float(smoothed[j][1])))
                if len(post_pts) >= 3:
                    break
            if len(pre_pts) < 2 or len(post_pts) < 2:
                continue
            pre_vy  = float(np.mean(np.diff([p[2] for p in pre_pts])))
            post_vy = float(np.mean(np.diff([p[2] for p in post_pts])))
            near_bounce = pre_vy >= GAP_BOUNCE_VEL_MIN and post_vy <= -GAP_BOUNCE_VEL_MIN
            if not near_bounce:
                continue
            bounce_frame = (gap_start + gap_end) // 2
            if bounce_frame - last_bounce_frame < BOUNCE_COOLDOWN_FRAMES:
                continue
            x_px = (float(pre_pts[-1][1]) + float(post_pts[0][1])) / 2
            y_px = max(float(pre_pts[-1][2]), float(post_pts[0][2]))
            candidates.append({
                'frame' : bounce_frame,
                'x_px'  : x_px,
                'y_px'  : y_px,
                'method': 'GAP',
            })
            last_bounce_frame = bounce_frame
        return candidates

    def _merge_candidates(self, vel_cands, gap_cands, parab_cands):
        PRIORITY = {'VEL': 0, 'GAP': 1, 'PARAB': 2}
        all_cands = sorted(vel_cands + gap_cands + parab_cands, key=lambda c: c['frame'])
        merged = []
        for cand in all_cands:
            if not merged:
                merged.append(cand)
                continue
            last = merged[-1]
            if cand['frame'] - last['frame'] < BOUNCE_COOLDOWN_FRAMES:
                if PRIORITY.get(cand['method'], 9) < PRIORITY.get(last['method'], 9):
                    merged[-1] = cand
            else:
                merged.append(cand)
        return merged

    def _vfit_on_window(self, window_pts):
        n = len(window_pts)
        if n < 4: return None
        frames = np.array([p[0] for p in window_pts], dtype=float)
        xs_px  = np.array([p[1] for p in window_pts], dtype=float)
        ys_px  = np.array([p[2] for p in window_pts], dtype=float)
        ys_work = ys_px
        peak_idx = int(np.argmax(ys_work))
        if peak_idx < 1 or peak_idx > n - 2:
            return None
        before_idx = list(range(0, peak_idx))
        after_idx  = list(range(peak_idx + 1, n))
        if len(before_idx) < 2 or len(after_idx) < 2:
            return None
        t_b = frames[before_idx]; y_b = ys_px[before_idx]
        t_a = frames[after_idx];  y_a = ys_px[after_idx]
        if float(np.mean(np.diff(y_b))) <= 0 or float(np.mean(np.diff(y_a))) >= 0:
            return None
        vy_t = (frames[:-1] + frames[1:]) / 2.0
        vy   = np.diff(ys_work)
        t_bounce = None
        near_mask = np.abs(vy_t - frames[peak_idx]) <= 3.0
        if near_mask.sum() < 2:
            near_mask = np.ones(len(vy_t), dtype=bool)
        try:
            vc = np.polyfit(vy_t[near_mask], vy[near_mask], 1)
            if abs(vc[0]) > 1e-9:
                t_cand = float(-vc[1] / vc[0])
                if float(frames[0]) <= t_cand <= float(frames[-1]):
                    t_bounce = t_cand
        except Exception:
            pass
        if t_bounce is None:
            pos_idx = np.where(vy > 0)[0]
            neg_idx = np.where(vy < 0)[0]
            if pos_idx.size and neg_idx.size:
                i0, i1 = int(pos_idx[-1]), int(neg_idx[0])
                t0, t1 = float(vy_t[i0]), float(vy_t[i1])
                v0, v1 = float(vy[i0]),   float(vy[i1])
                if t1 > t0 and abs(v1 - v0) > 1e-9:
                    t_bounce = t0 + (-v0) * (t1 - t0) / (v1 - v0)
        if t_bounce is None or not (float(frames[0]) <= t_bounce <= float(frames[-1])):
            return None
        deg_b = 2 if len(before_idx) >= 3 else 1
        deg_a = 2 if len(after_idx)  >= 3 else 1
        try:
            c_b = np.polyfit(t_b, y_b, deg_b)
            c_a = np.polyfit(t_a, y_a, deg_a)
            y_bounce = (float(np.polyval(c_b, t_bounce)) +
                        float(np.polyval(c_a, t_bounce))) / 2.0
        except Exception:
            y_bounce = float(np.interp(t_bounce, frames, ys_px))
        try:
            c_x      = np.polyfit(frames, xs_px, 1)
            x_bounce = float(np.polyval(c_x, t_bounce))
        except Exception:
            x_bounce = float(np.interp(t_bounce, frames, xs_px))
        return {'frame': int(round(t_bounce)), 'x_px': x_bounce, 'y_px': y_bounce}

    def _raw_window(self, frame, fallback_smoothed):
        src = self.raw_positions if self.raw_positions is not None else fallback_smoothed
        pts = []
        lo = max(0, frame - CONTACT_FIT_WINDOW)
        hi = min(len(src), frame + CONTACT_FIT_WINDOW + 1)
        for i in range(lo, hi):
            pt = src[i]
            if pt is not None:
                pts.append((i, float(pt[0]), float(pt[1])))
        return pts

    def _contact_fit_near_frame(self, frame, fallback_smoothed):
        pts = self._raw_window(frame, fallback_smoothed)
        if len(pts) < 4:
            return None
        frames = np.array([p[0] for p in pts], dtype=float)
        xs_px  = np.array([p[1] for p in pts], dtype=float)
        ys_px  = np.array([p[2] for p in pts], dtype=float)
        vy = np.diff(ys_px)
        if len(vy) < 2:
            return None
        sign_pairs = np.where((vy[:-1] > 0) & (vy[1:] <= 0))[0]
        if sign_pairs.size:
            local = int(sign_pairs[np.argmin(np.abs(frames[sign_pairs + 1] - frame))])
            before_last = local + 1
            after_first = local + 2
        else:
            peak_idx = int(np.argmax(ys_px))
            if peak_idx < 1 or peak_idx >= len(pts) - 1:
                return None
            before_last = peak_idx
            after_first = peak_idx + 1
        pre_idx = np.arange(max(0, before_last - 3), before_last + 1)
        post_idx = np.arange(after_first, min(len(pts), after_first + 4))
        if len(pre_idx) < 2 or len(post_idx) < 2:
            return None
        if float(np.mean(np.diff(ys_px[pre_idx]))) <= 0:
            return None
        if float(np.mean(np.diff(ys_px[post_idx]))) >= 0:
            return None
        t0 = float(frames[before_last]); t1 = float(frames[after_first])
        if t1 <= t0:
            return None
        v0 = float(vy[max(0, before_last - 1)])
        v1 = float(vy[min(len(vy) - 1, after_first - 1)])
        t_bounce = t0 + (-v0) * (t1 - t0) / (v1 - v0) if abs(v1 - v0) > 1e-9 else 0.5 * (t0 + t1)
        t_bounce = float(np.clip(t_bounce, t0, t1))
        try:
            x_bounce = float(np.polyval(np.polyfit(frames, xs_px, 1), t_bounce))
        except Exception:
            x_bounce = float(np.interp(t_bounce, frames, xs_px))
        try:
            c_pre = np.polyfit(frames[pre_idx], ys_px[pre_idx], 2 if len(pre_idx) >= 3 else 1)
            c_post = np.polyfit(frames[post_idx], ys_px[post_idx], 2 if len(post_idx) >= 3 else 1)
            y_bounce = max(float(np.polyval(c_pre, t_bounce)), float(np.polyval(c_post, t_bounce)))
        except Exception:
            y_bounce = float(np.interp(t_bounce, frames, ys_px))
        return {'frame': int(round(t_bounce)), 'x_px': x_bounce, 'y_px': y_bounce}

def draw_trail(frame, trail):
    n = len(trail)
    for i, pt in enumerate(trail):
        if pt is None: continue
        t=((i+1)/n); r=max(4,int(10*t)); a=0.25+0.70*t
        color=(0,int(180+75*t),int(160+95*t))
        ov=frame.copy(); cv2.circle(ov,pt,r,color,-1,cv2.LINE_AA)
        cv2.addWeighted(ov,a,frame,1-a,0,frame)

def draw_court_overlay(frame, kps):
    for (i,j) in SKELETON:
        cv2.line(frame,(int(kps[i][0]),int(kps[i][1])),
                       (int(kps[j][0]),int(kps[j][1])),(255,255,255),1,cv2.LINE_AA)
    for k in range(12):
        x,y=int(kps[k][0]),int(kps[k][1])
        cv2.circle(frame,(x,y),5,(0,200,255),-1,cv2.LINE_AA)

def draw_hud(frame, frame_idx, total, source, fps_proc, last_decision):
    colors={'BOTH':(0,255,128),'YOLO':(0,255,0),'TRACKNET':(255,255,0),'NONE':(0,80,255)}
    color=colors.get(source,(200,200,200))
    ov=frame.copy(); cv2.rectangle(ov,(0,0),(frame.shape[1],38),(0,0,0),-1)
    cv2.addWeighted(ov,0.5,frame,0.5,0,frame)
    fps_str=f"{fps_proc:.1f}fps" if fps_proc>0 else "--fps"
    text=f"[{source}] Frame {frame_idx+1}/{total} | {fps_str}"
    if last_decision: text+=f"  Last: {last_decision}"
    cv2.putText(frame,text,(10,26),cv2.FONT_HERSHEY_SIMPLEX,0.60,color,2,cv2.LINE_AA)

def draw_court_boundary(frame, kps):
    polygon = build_court_polygon(kps)
    if polygon is None: return
    pts = polygon.astype(np.int32).reshape((-1, 1, 2))
    cv2.polylines(frame, [pts], isClosed=True,
                  color=(0, 255, 128), thickness=2, lineType=cv2.LINE_AA)

def draw_inout_label(frame, decision, bounce_px, frames_left, total_display):
    if decision is None or bounce_px is None: return
    color_map = {'IN': (0,255,0), 'OUT': (0,0,255), 'UNCERTAIN': (0,200,255)}
    color = color_map.get(decision, (128,128,128))
    alpha = max(0.3, frames_left/total_display)
    bx, by = int(bounce_px[0]), int(bounce_px[1])
    ov = frame.copy(); cv2.circle(ov,(bx,by),14,color,-1,cv2.LINE_AA)
    cv2.addWeighted(ov,alpha,frame,1-alpha,0,frame)
    cv2.circle(frame,(bx,by),14,(255,255,255),2,cv2.LINE_AA)
    font=cv2.FONT_HERSHEY_SIMPLEX; scale=1.4; thick=3
    (tw,th),_=cv2.getTextSize(decision,font,scale,thick)
    tx,ty=bx-tw//2,by-30
    ov2=frame.copy(); cv2.rectangle(ov2,(tx-6,ty-th-6),(tx+tw+6,ty+6),color,-1)
    cv2.addWeighted(ov2,alpha,frame,1-alpha,0,frame)
    cv2.putText(frame,decision,(tx,ty),font,scale,(255,255,255),thick,cv2.LINE_AA)
    if frames_left > total_display*0.5:
        H_f,W_f=frame.shape[:2]; bs=3.5; bt=6
        (bw,bh),_=cv2.getTextSize(decision,font,bs,bt)
        bx2,by2=W_f//2-bw//2,H_f//2+bh//2
        bg=frame.copy(); pad=20
        cv2.rectangle(bg,(bx2-pad,by2-bh-pad),(bx2+bw+pad,by2+pad),color,-1)
        cv2.addWeighted(bg,0.55,frame,0.45,0,frame)
        cv2.putText(frame,decision,(bx2,by2),font,bs,(255,255,255),bt,cv2.LINE_AA)

print("✅ All helper functions defined")
print("   BounceDetector : 3-pass detection | configurable VFIT/CONTACTFIT | configurable decision")

✅ All helper functions defined
   BounceDetector : 3-pass detection | configurable VFIT/CONTACTFIT | configurable decision


In [ ]:
print(f'Loading TrackNetV5 5-frame: {TRACKNET_PATH}')
ckpt     = torch.load(TRACKNET_PATH, map_location=device)
tn_model = TrackNetV5_5frame().to(device)
tn_model.load_state_dict(ckpt['model_state'])
tn_model.eval()
print(f'  ✅ TrackNetV5_5frame  '
      f'best_f1={ckpt.get("best_f1","N/A")}  '
      f'epoch={ckpt.get("epoch","N/A")}  '
      f'n_frames={ckpt.get("n_frames","N/A")}')

print(f'Loading PickleballCourtNet: {COURTNET_PATH}')
court_model = PickleballCourtNet(out_channels=13).to(device)
court_model.load_state_dict(torch.load(COURTNET_PATH, map_location=device))
court_model.eval()
print(f'  ✅ PickleballCourtNet')

print(f'\n✅ Both models ready on {device}')

Loading TrackNetV5 5-frame: /content/drive/MyDrive/In_Out_Pickleball/tracknetv5_5frame_scratch/checkpoints/best_tracknetv5_5frame.pth
  ✅ TrackNetV5_5frame  best_f1=0.97340930674264  epoch=18  n_frames=5
Loading PickleballCourtNet: /content/drive/MyDrive/In_Out_Pickleball/TennisCourtNet_v2/best.pt
  ✅ PickleballCourtNet

✅ Both models ready on cuda


In [ ]:
cap    = cv2.VideoCapture(VIDEO_PATH)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if MAX_FRAMES: total = min(total, MAX_FRAMES)
print(f'Video: {orig_w}x{orig_h} @ {fps:.1f}fps | {total} frames')

print('\nPhase 1: Buffering frames...')
all_frames = []
while len(all_frames) < total:
    ret, frame = cap.read()
    if not ret: break
    all_frames.append(frame)
cap.release()
N = len(all_frames)
print(f'  {N} frames buffered')

Video: 1920x1080 @ 30.0fps | 255 frames

Phase 1: Buffering frames...
  255 frames buffered


In [ ]:
print('Phase 2: TrackNetV5 5-frame inference...')
print(f'  Context: ±{TN_HALF} frames  |  boundary frames clamped (all {"{N}"} frames processed)')
tn_results = [None]*N
last_cx = last_cy = None

preprocessed = [preprocess_frame_tn(f) for f in all_frames]
t_tn = time.time()

with torch.no_grad():
    for i in range(N):
        quintuplet = torch.cat(
            [preprocessed[max(0, min(N-1, i+d))] for d in range(-TN_HALF, TN_HALF+1)],
            dim=0
        ).unsqueeze(0).to(device)   # 1×15×288×512

        heatmap = tn_model(quintuplet)
        center  = extract_ball_center(heatmap, TN_THRESH)

        if center is not None:
            cx_o, cy_o = model_to_orig(*center, orig_w, orig_h)
            if last_cx is not None:
                if np.hypot(cx_o - last_cx, cy_o - last_cy) > TN_MAX_TRAVEL:
                    center = None

        if center is not None:
            tn_results[i] = (cx_o, cy_o)
            last_cx, last_cy = cx_o, cy_o
        else:
            last_cx = last_cy = None

        if (i+1) % 100 == 0 or i == N - 1:
            det = sum(r is not None for r in tn_results[:i+1])
            print(f'  TN {i+1}/{N} | det:{det} ({det/(i+1)*100:.1f}%)')

del preprocessed

det_total = sum(r is not None for r in tn_results)
print(f'  TrackNet done in {time.time()-t_tn:.1f}s')
print(f'  Detections: {det_total}/{N} ({det_total/N*100:.1f}%)')

print('Phase 3: Smoothing...')
raw_xs = [float(r[0]) if r is not None else np.nan for r in tn_results]
raw_ys = [float(r[1]) if r is not None else np.nan for r in tn_results]
smooth_xs = smooth_coords(raw_xs); smooth_ys = smooth_coords(raw_ys)
smoothed = []
for i in range(N):
    if tn_results[i] is not None:
        smoothed.append((int(smooth_xs[i]), int(smooth_ys[i])))
    else:
        lo = max(0, i-SMOOTH_MAX_GAP); hi = min(N, i+SMOOTH_MAX_GAP+1)
        nearby = any(tn_results[j] is not None for j in range(lo, hi) if j!=i)
        if nearby and not np.isnan(raw_xs[i]):
            smoothed.append((int(smooth_xs[i]), int(smooth_ys[i])))
        else:
            smoothed.append(None)
print('  Smoothing done')

Phase 2: TrackNetV5 5-frame inference...
  Context: ±2 frames  |  boundary frames clamped (all {N} frames processed)
  TN 100/255 | det:98 (98.0%)
  TN 200/255 | det:194 (97.0%)
  TN 255/255 | det:209 (82.0%)
  TrackNet done in 15.3s
  Detections: 209/255 (82.0%)
Phase 3: Smoothing...
  Smoothing done


In [ ]:
print("Phase 6: Court keypoint detection...")
stabilizer     = KPStabilizer(COURT_BUFFER_SIZE, COURT_MOVE_THRESH, COURT_EMA_ALPHA)
court_kps_all    = [None]*N
court_Hmat_all   = [None]*N
court_Hpx2ct_all = [None]*N
latest_H         = None
latest_H_px2ct   = None

for i in range(0, N, COURT_DETECT_EVERY):
    kps, H_mat  = predict_court_kps(court_model, all_frames[i], device)
    stable_kps  = stabilizer.update(kps)
    if H_mat is not None: latest_H = H_mat
    H_px2ct = build_pixel_to_court_homography(stable_kps)
    if H_px2ct is not None: latest_H_px2ct = H_px2ct
    for j in range(i, min(i+COURT_DETECT_EVERY, N)):
        court_kps_all[j]    = stable_kps.copy()
        court_Hmat_all[j]   = latest_H
        court_Hpx2ct_all[j] = latest_H_px2ct
    if (i//COURT_DETECT_EVERY+1)%10 == 0:
        print(f"  Court {i}/{N}...")

n_px2ct = sum(h is not None for h in court_Hpx2ct_all)
print(f"✅ Court detected on {N//COURT_DETECT_EVERY} keyframes  | px2ct H: {n_px2ct}/{N} frames")

Phase 6: Court keypoint detection...
  Court 90/255...
  Court 190/255...
✅ Court detected on 25 keyframes  | px2ct H: 255/255 frames


In [ ]:
print("Phase 5: Bounce detection + IN/OUT (safe hybrid mode)...")
contact_corrector = load_contact_corrector()
print(f"  Contact correction: {'ON' if contact_corrector is not None else 'OFF'}")
print(f"  Refine mode: {BOUNCE_REFINE_MODE} | Decision mode: {DECISION_MODE}")
bounce_detector = BounceDetector(raw_positions=tn_results, contact_corrector=contact_corrector)
bounce_events   = bounce_detector.detect_bounces(
    smoothed, court_kps_all, court_Hpx2ct_all)

in_count      = sum(1 for e in bounce_events if e['decision'] == 'IN')
out_count     = sum(1 for e in bounce_events if e['decision'] == 'OUT')
unc_count     = sum(1 for e in bounce_events if e['decision'] == 'UNCERTAIN')
unknown_count = sum(1 for e in bounce_events if e['decision'] == 'UNKNOWN')

for ev in bounce_events:
    dist_str = f"{ev['dist_px']:.1f}px" if ev.get('dist_px') is not None else "   N/A"
    xcm_str  = f"{ev['x_cm']:.1f}" if ev.get('x_cm') is not None else "N/A"
    ycm_str  = f"{ev['y_cm']:.1f}" if ev.get('y_cm') is not None else "N/A"
    print(f"  Bounce @ frame {ev['frame']:>5}: "
          f"px=({ev['x_px']:>6.0f},{ev['y_px']:>6.0f})  "
          f"dist={dist_str:>9}  "
          f"court=({xcm_str:>7}cm,{ycm_str:>7}cm)  "
          f"→ {ev['decision']:>9}  [{ev['method']}]")

print(f"\n✅ {len(bounce_events)} bounces detected")
print(f"   IN:{in_count}  OUT:{out_count}  UNCERTAIN:{unc_count}  UNKNOWN:{unknown_count}")
print(f"   (UNCERTAIN = within {POLYGON_MARGIN_PX}px of pixel court boundary, or calibrated near-line in COURT_CM mode)")
print(f"   (UNKNOWN   = court polygon unavailable or ball too high above court)")

print("\n── VY DEBUG (every 10 frames, showing local vy and peak candidates) ──")
positions_dbg = [
    (i, float(smoothed[i][0]), float(smoothed[i][1]))
    for i in range(N) if smoothed[i] is not None
]
if len(positions_dbg) > 4:
    ys_dbg  = np.array([p[2] for p in positions_dbg], dtype=float)
    fr_dbg  = np.array([p[0] for p in positions_dbg], dtype=float)
    vy_dbg  = np.diff(ys_dbg)
    print(f"  Total tracked positions: {len(positions_dbg)}  |  vy range: [{vy_dbg.min():.2f}, {vy_dbg.max():.2f}]  |  BOUNCE_VEL_MIN_PX={BOUNCE_VEL_MIN_PX}")
    print(f"  Sample vy values (every 5 frames): ", end='')
    print(', '.join(f'{v:.1f}' for v in vy_dbg[::5]))
    print(f"\n  Local Y maxima (potential bounce peaks, threshold-free):")
    for j in range(2, len(vy_dbg) - 2):
        avg_b = float(np.mean(vy_dbg[max(0,j-2):j]))
        avg_a_arr = vy_dbg[j+1:j+3]
        avg_a = float(np.mean(avg_a_arr)) if len(avg_a_arr) > 0 else float('nan')
        delta = avg_b - avg_a
        frame_n = int(fr_dbg[j+1]) if j+1 < len(fr_dbg) else '?'
        is_sign_flip = vy_dbg[j-1] > 0 and vy_dbg[j] <= 0
        is_decel = (avg_b >= BOUNCE_VEL_MIN_PX * 8
                    and avg_a >= 0 and avg_a < BOUNCE_VEL_MIN_PX
                    and delta >= BOUNCE_VEL_MIN_PX * 8)
        if not (is_sign_flip or is_decel):
            continue
        bounce_type = 'DECEL' if (is_decel and not is_sign_flip) else 'SIGN-FLIP'
        primary = avg_b >= BOUNCE_VEL_MIN_PX and avg_a <= -BOUNCE_VEL_MIN_PX
        fallback = avg_b > 0 and not np.isnan(avg_a) and avg_a < 0 and delta >= BOUNCE_VEL_MIN_PX
        decel_ok = (avg_b >= BOUNCE_VEL_MIN_PX * 8 and avg_a >= 0
                    and avg_a < BOUNCE_VEL_MIN_PX and delta >= BOUNCE_VEL_MIN_PX * 8)
        status = 'DETECTED' if (primary or fallback or decel_ok) else f'MISSED(avg_b={avg_b:.2f} avg_a={avg_a:.2f})'
        print(f"    [{bounce_type}] frame~{frame_n:>5}  vy_before={avg_b:+.2f}  vy_after={avg_a:+.2f}  → {status}')")

last_ev_frame = bounce_events[-1]['frame'] if bounce_events else 0
focus_start   = max(0, last_ev_frame - 20)
focus_end     = min(N, last_ev_frame + 120)

focus_pos = [
    (i, float(smoothed[i][0]), float(smoothed[i][1]))
    for i in range(focus_start, focus_end)
    if i < N and smoothed[i] is not None
]
print(f"\n── LATE-VIDEO DEBUG (frames {focus_start}–{focus_end}) ──")
print(f"  Last detected bounce: frame {last_ev_frame}")
print(f"  Smoothed positions in window: {len(focus_pos)}")
for p in focus_pos:
    print(f"    frame {p[0]:>5}: x={p[1]:>7.1f}  y={p[2]:>7.1f}")

if len(focus_pos) >= 3:
    ys_f = np.array([p[2] for p in focus_pos], dtype=float)
    fr_f = np.array([p[0] for p in focus_pos], dtype=float)
    vy_f = np.diff(ys_f)
    print(f"  vy values (per consecutive detection):")
    for k in range(len(vy_f)):
        f0, f1 = int(fr_f[k]), int(fr_f[k+1])
        print(f"    frames {f0:>5}→{f1:>5}  vy={vy_f[k]:+.2f}  (gap={f1-f0} frames)")
    print(f"  Sign flips (potential bounce peaks):")
    found = False
    for k in range(1, len(vy_f)):
        if vy_f[k-1] > 0 and vy_f[k] <= 0:
            avg_b = float(np.mean(vy_f[max(0,k-2):k]))
            avg_a = float(np.mean(vy_f[k+1:k+3])) if k+1 < len(vy_f) else float('nan')
            fn = int(fr_f[k+1]) if k+1 < len(fr_f) else '?'
            thresh_ok = avg_b >= BOUNCE_VEL_MIN_PX and avg_a <= -BOUNCE_VEL_MIN_PX
            fallback_ok = avg_b > 0 and not np.isnan(avg_a) and avg_a < 0 and (avg_b - avg_a) >= BOUNCE_VEL_MIN_PX
            reason = 'DETECTED' if (thresh_ok or fallback_ok) else (
                'NO_AFTER_DATA' if np.isnan(avg_a) else
                f'BELOW_THRESH(need>={BOUNCE_VEL_MIN_PX:.1f}, got avg_b={avg_b:.2f} avg_a={avg_a:.2f})')
            print(f"    frame~{fn:>5}  avg_before={avg_b:+.2f}  avg_after={avg_a:+.2f}  → {reason}")
            found = True
    if not found:
        print(f"    (no vy sign flip in this window — no Y-max/bounce peak found)")
        print(f"    vy_min={vy_f.min():.2f}  vy_max={vy_f.max():.2f}  all going {'DOWN (vy>0)' if vy_f.mean()>0 else 'UP (vy<0)' if vy_f.mean()<0 else 'mixed'}")
else:
    print(f"  Not enough positions to compute vy (need ≥3, have {len(focus_pos)})")
    print(f"  → Ball likely left frame immediately after last bounce — undetectable with velocity reversal method")

Phase 5: Bounce detection + IN/OUT (safe hybrid mode)...
  Contact correction: OFF
  Refine mode: VFIT | Decision mode: PIXEL_POLYGON
  Bounce @ frame    16: px=(   735,   349)  dist=   18.5px  court=(  142.2cm, 1302.6cm)  →        IN  [VEL+VFIT]
  Bounce @ frame    59: px=(  1276,  1017)  dist=   18.9px  court=(  426.4cm,   11.7cm)  →        IN  [VEL+VFIT]
  Bounce @ frame    66: px=(  1400,   932)  dist=  104.1px  court=(  482.6cm,  105.0cm)  →        IN  [VEL+POS]
  Bounce @ frame    94: px=(   699,   364)  dist=   33.5px  court=(  120.2cm, 1249.2cm)  →        IN  [VEL+VFIT]
  Bounce @ frame   131: px=(  1363,   763)  dist=  229.8px  court=(  490.0cm,  329.1cm)  →        IN  [VEL+POS]
  Bounce @ frame   140: px=(  1556,   825)  dist=   89.3px  court=(  564.6cm,  241.1cm)  →        IN  [VEL+VFIT]
  Bounce @ frame   170: px=(  1337,   326)  dist=   -4.7px  court=(  576.1cm, 1389.2cm)  → UNCERTAIN  [VEL+VFIT]
  Bounce @ frame   207: px=(  1892,   751)  dist= -241.2px  court=(  734.5cm,

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

cm_traj = []
for i in range(N):
    pt = smoothed[i]; H = court_Hpx2ct_all[i]
    if pt is None or H is None: continue
    x_cm, y_cm = pixel_to_court_cm(pt[0], pt[1], H)
    if x_cm is not None: cm_traj.append((i, x_cm, y_cm))

color_map = {'IN': 'lime', 'OUT': 'red', 'UNCERTAIN': 'gold', 'UNKNOWN': 'grey'}

fig, axes = plt.subplots(4, 1, figsize=(20, 16))

if cm_traj:
    frames_cm = [p[0] for p in cm_traj]
    xs_cm     = [p[1] for p in cm_traj]
    ys_cm     = [p[2] for p in cm_traj]
    axes[0].plot(frames_cm, xs_cm, 'b-', lw=1, label='x_cm (via homography)')
    axes[0].axhline(COURT_X_MIN, color='r', ls='--', alpha=0.5, label='boundary')
    axes[0].axhline(COURT_X_MAX, color='r', ls='--', alpha=0.5)
    for ev in bounce_events:
        axes[0].axvline(ev['frame'], color=color_map.get(ev['decision'],'grey'), alpha=0.7, lw=1.5)
axes[0].set_title('Ball X court space (cm) — via homography, for reference only')
axes[0].legend(); axes[0].grid(alpha=0.3)

if cm_traj:
    axes[1].plot(frames_cm, ys_cm, 'g-', lw=1, label='y_cm (via homography)')
    axes[1].axhline(COURT_Y_MIN, color='r', ls='--', alpha=0.5)
    axes[1].axhline(COURT_Y_MAX, color='r', ls='--', alpha=0.5)
    for ev in bounce_events:
        axes[1].axvline(ev['frame'], color=color_map.get(ev['decision'],'grey'), alpha=0.7, lw=2)
axes[1].set_title('Ball Y court space (cm)  lime=IN  red=OUT  gold=UNCERTAIN')
axes[1].legend(); axes[1].grid(alpha=0.3)

frames_ev = [e['frame']    for e in bounce_events if e.get('dist_px') is not None]
dists_ev  = [e['dist_px']  for e in bounce_events if e.get('dist_px') is not None]
colors_ev = [color_map.get(e['decision'],'grey') for e in bounce_events if e.get('dist_px') is not None]
axes[2].bar(frames_ev, dists_ev, color=colors_ev, alpha=0.8, width=3)
axes[2].axhline( POLYGON_MARGIN_PX,  color='orange', ls='--', lw=1.5, label=f'+{POLYGON_MARGIN_PX}px (IN threshold)')
axes[2].axhline(-POLYGON_MARGIN_PX,  color='orange', ls='--', lw=1.5, label=f'-{POLYGON_MARGIN_PX}px (OUT threshold)')
axes[2].axhline(0, color='white', lw=0.5)
axes[2].set_title('Signed pixel distance from court boundary (+ = inside, − = outside)')
axes[2].legend(); axes[2].grid(alpha=0.3)

raw_ys = [smoothed[i][1] if smoothed[i] is not None else None for i in range(N)]
frames_ball = [i for i in range(N) if raw_ys[i] is not None]
ys_ball     = [raw_ys[i] for i in frames_ball]
if frames_ball:
    axes[3].plot(frames_ball, ys_ball, 'c-', lw=1, label='Y pixel (smoothed)')
    for ev in bounce_events:
        axes[3].axvline(ev['frame'], color=color_map.get(ev['decision'],'grey'), alpha=0.7, lw=2)
        axes[3].plot(ev['x_px'] if False else ev['frame'], ev['y_px'],
                     'o', color=color_map.get(ev['decision'],'grey'), ms=8, zorder=5)
axes[3].set_title('Ball Y pixel trajectory  (bounce markers: contact fit or fallback)')
axes[3].legend(); axes[3].grid(alpha=0.3)
axes[3].set_xlabel('Frame')

patches = [mpatches.Patch(color=v, label=k) for k,v in color_map.items()]
plt.suptitle(
    f'New Algorithm Diagnostic | {len(bounce_events)} bounces  '
    f'IN:{in_count}  OUT:{out_count}  UNCERTAIN:{unc_count}  UNKNOWN:{unknown_count}',
    fontsize=11)
fig.legend(handles=patches, loc='upper right', fontsize=9)
plt.tight_layout()

diag_path = OUTPUT_PATH.replace('.mp4', '_diagnostic.png')
plt.savefig(diag_path, dpi=120); plt.show()
print(f"\u2705 Diagnostic saved \u2192 {diag_path}")

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'),
                          fps, (orig_w, orig_h))
print(f'Phase 6: Rendering \u2192 {OUTPUT_PATH}')

trail          = deque(maxlen=TRAIL_LEN)
last_decision  = None
active_bounce  = None
t_start        = time.time()
bounce_by_frame= {e['frame']: e for e in bounce_events}

for i in range(N):
    frame  = all_frames[i].copy()
    pt     = smoothed[i]
    source = 'TRACKNET' if tn_results[i] is not None else 'NONE'
    trail.append(pt)

    if i in bounce_by_frame:
        ev            = bounce_by_frame[i]
        last_decision = ev['decision']
        active_bounce = {
            'x_px'      : ev['x_px'],
            'y_px'      : ev['y_px'],
            'decision'  : ev['decision'],
            'frames_left': BOUNCE_DISPLAY_FRAMES,
        }

    if court_kps_all[i] is not None:
        draw_court_overlay(frame, court_kps_all[i])
        draw_court_boundary(frame, court_kps_all[i])

    draw_trail(frame, list(trail))

    if tn_results[i] is not None:
        cx, cy = tn_results[i]
        cv2.circle(frame,(cx,cy),13,(255,255,0),2,cv2.LINE_AA)
        cv2.putText(frame,'TN',(cx+16,cy-4),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,0),1)

    if pt is not None:
        cv2.circle(frame, pt, 8, (0,255,255), -1, cv2.LINE_AA)

    if active_bounce is not None and active_bounce['frames_left'] > 0:
        draw_inout_label(frame, active_bounce['decision'],
                          (active_bounce['x_px'], active_bounce['y_px']),
                          active_bounce['frames_left'], BOUNCE_DISPLAY_FRAMES)
        active_bounce['frames_left'] -= 1

    fps_proc = (i+1)/(time.time()-t_start)
    draw_hud(frame, i, N, source, fps_proc, last_decision)
    writer.write(frame)
    if (i+1)%100==0 or i==N-1:
        print(f'  Rendered {i+1}/{N}', end='\r')

writer.release()
total_time = time.time() - t_start
print(f'\n\n{"="*65}')
print(f'  \u2705 DONE')
print(f'{"="*65}')
print(f'  Output    : {OUTPUT_PATH}')
print(f'  Frames    : {N}')
print(f'  Bounces   : {len(bounce_events)}  IN:{in_count}  OUT:{out_count}  '
      f'UNCERTAIN:{unc_count}  UNKNOWN:{unknown_count}')
print(f'  Algorithm : VSign + PARAB + GAP → raw contact fit → court-cm decision')
print(f'  Time      : {total_time:.1f}s')
print(f'{"="*65}')

Phase 6: Rendering → /content/drive/MyDrive/In_Out_Pickleball/result_in_out/first_codex_ver2_new_algorithm_output_accurate.mp4
  Rendered 255/255

  ✅ DONE
  Output    : /content/drive/MyDrive/In_Out_Pickleball/result_in_out/first_codex_ver2_new_algorithm_output_accurate.mp4
  Frames    : 255
  Bounces   : 10  IN:7  OUT:1  UNCERTAIN:1  UNKNOWN:1
  Algorithm : VSign + PARAB + GAP → raw contact fit → court-cm decision
  Time      : 14.8s


In [ ]:
log_path = OUTPUT_PATH.replace('.mp4', '_bounce_log.json')
with open(log_path, 'w') as f:
    json.dump(bounce_events, f, indent=2)
print(f"\u2705 Bounce log \u2192 {log_path}")
print(f"\n{'\u2500'*75}")
print(f"  {'Frame':>6}  {'x_px':>7}  {'y_px':>7}  {'dist_px':>8}  "
      f"{'x_cm':>8}  {'y_cm':>8}  Decision   Method")
print(f"{'\u2500'*75}")
for ev in bounce_events:
    dist_s = f"{ev['dist_px']:.1f}" if ev.get('dist_px') is not None else "N/A"
    xcm_s  = f"{ev['x_cm']:.1f}"  if ev.get('x_cm')  is not None else "N/A"
    ycm_s  = f"{ev['y_cm']:.1f}"  if ev.get('y_cm')  is not None else "N/A"
    print(f"  {ev['frame']:>6}  {ev['x_px']:>7.0f}  {ev['y_px']:>7.0f}  "
          f"{dist_s:>8}  {xcm_s:>8}  {ycm_s:>8}  "
          f"{ev['decision']:>9}  {ev.get('method','?')}")